
# Mushroom44 — Step 0.6: Handle 23 Exact Duplicate Images

Notebook này dùng để **xử lý 23 cặp ảnh trùng tuyệt đối (SHA256 giống hệt)** đã được phát hiện ở Step 0.

## Mục tiêu
- **Không chỉnh sửa dataset gốc** trong `/kaggle/input/...`.
- Tạo ra **một phiên bản deduped mới** để dùng cho nghiên cứu và huấn luyện tiếp theo.
- Xuất đầy đủ báo cáo: danh sách ảnh giữ lại, ảnh loại bỏ, phân bố lớp mới, và file ZIP tổng hợp.

## Quy ước xử lý
- Chỉ xử lý **exact duplicates** trong `exact_duplicates_all.csv`.
- Vì các cặp trùng đều **nằm trong cùng split** và **cùng nhãn**, ta sẽ:
  - **Giữ 1 ảnh đại diện** trong mỗi nhóm SHA256.
  - **Loại các ảnh trùng còn lại** khỏi phiên bản dataset nghiên cứu mới.
- Cặp ảnh gần trùng train/test với `pHash = 4` đã được bạn xem xét và **không coi là cùng ảnh**, do đó **không xử lý trong notebook này**.

## Đầu ra chính
- `dedup_plan.csv`
- `removed_files.csv`
- `kept_files.csv`
- `class_distribution_deduped.csv`
- `dedup_summary.json`
- thư mục dataset mới: `mushroom-44-clean-deduped/`
- file ZIP tổng hợp: `Mushroom44_Step0_6_DedupExact.zip`


In [1]:

# =========================
# 1. CÀI ĐẶT & IMPORT
# =========================
import os
import json
import zipfile
import shutil
from pathlib import Path
from collections import defaultdict

import pandas as pd
import numpy as np
from PIL import Image
from IPython.display import display

pd.set_option('display.max_colwidth', 200)
pd.set_option('display.max_columns', 50)


In [18]:

from pathlib import Path
import zipfile

# ============================================================
# CẤU HÌNH AUDIT INPUT
# ============================================================

AUDIT_INPUT = Path(
    '/kaggle/input/datasets/thoduynhunh/mushroom44-step0-datasetaudit'
)

required = {
    'exact_duplicates_all.csv',
    'audit_summary.json',
    'dataset_manifest.csv',
    'class_distribution.csv',
    'class_to_idx.json'
}

assert AUDIT_INPUT.exists(), f'Không tồn tại: {AUDIT_INPUT}'

# ============================================================
# TỰ ĐỘNG XÁC ĐỊNH ZIP
# ============================================================

if AUDIT_INPUT.is_file():
    AUDIT_ZIP = AUDIT_INPUT

else:
    print('Đã nhận diện thư mục Kaggle Dataset.')

    zip_files = sorted(AUDIT_INPUT.rglob('*.zip'))

    # Tìm ZIP có đủ các file yêu cầu ở thư mục gốc
    AUDIT_ZIP = None

    for candidate in zip_files:
        with zipfile.ZipFile(candidate) as z:
            if required.issubset(set(z.namelist())):
                AUDIT_ZIP = candidate
                break

    if AUDIT_ZIP is None:
        print('Không có ZIP phù hợp. Kiểm tra file đã giải nén...')

        files = [
            p for p in AUDIT_INPUT.rglob('*')
            if p.is_file() and p.suffix.lower() != '.zip'
        ]

        file_map = {}
        for p in files:
            if p.name in file_map:
                raise ValueError(
                    f'Trùng tên file khi đóng gói: {p.name}'
                )
            file_map[p.name] = p

        missing = required - set(file_map)

        assert not missing, (
            f'Thiếu các file Step 0: {missing}'
        )

        # Tạo ZIP mới trong working (không sửa input)
        AUDIT_ZIP = Path(
            '/kaggle/working/Mushroom44_Step0_DatasetAudit.zip'
        )

        with zipfile.ZipFile(
            AUDIT_ZIP, 'w',
            compression=zipfile.ZIP_DEFLATED
        ) as z:
            for name, path in file_map.items():
                z.write(path, arcname=name)

        print('Đã tạo ZIP từ dữ liệu Step 0.')

# ============================================================
# XÁC MINH
# ============================================================

assert DATASET_ROOT.exists(), 'Không tìm thấy dataset gốc.'
assert AUDIT_ZIP.is_file(), 'AUDIT_ZIP không phải file.'

with zipfile.ZipFile(AUDIT_ZIP) as z:
    names = set(z.namelist())
    missing = required - names

    assert not missing, f'Thiếu file: {missing}'

    print('\nCác file ZIP:')
    for name in sorted(names):
        print('  ', name)

print('\nKIỂM TRA THÀNH CÔNG!')
print('DATASET_ROOT =', DATASET_ROOT)
print('AUDIT_ZIP    =', AUDIT_ZIP)
print('WORK_ROOT    =', WORK_ROOT)


Đã nhận diện thư mục Kaggle Dataset.
Không có ZIP phù hợp. Kiểm tra file đã giải nén...
Đã tạo ZIP từ dữ liệu Step 0.

Các file ZIP:
   audit_summary.json
   class_distribution.csv
   class_distribution.png
   class_to_idx.json
   corrupted_images.csv
   dataset_manifest.csv
   exact_duplicates_all.csv
   exact_duplicates_cross_split.csv
   image_audit_checkpoint.csv
   image_size_outliers.csv
   near_duplicates_cross_split.csv
   resolution_distribution.png
   source_specimen_leakage.csv
   unexpected_files.csv

KIỂM TRA THÀNH CÔNG!
DATASET_ROOT = /kaggle/input/datasets/thoduynhunh/mushroom-44-clean
AUDIT_ZIP    = /kaggle/working/Mushroom44_Step0_DatasetAudit.zip
WORK_ROOT    = /kaggle/working/Mushroom44_Step0_6_DedupExact


In [19]:

# =========================
# 3. KIỂM TRA ĐẦU VÀO
# =========================
assert DATASET_ROOT.exists(), f'Không tìm thấy DATASET_ROOT: {DATASET_ROOT}'
assert AUDIT_ZIP.exists(), f'Không tìm thấy AUDIT_ZIP: {AUDIT_ZIP}'

with zipfile.ZipFile(AUDIT_ZIP) as z:
    names = set(z.namelist())
    required = {
        'exact_duplicates_all.csv',
        'audit_summary.json',
        'dataset_manifest.csv',
        'class_distribution.csv',
        'class_to_idx.json'
    }
    missing = required - names
    assert not missing, f'Thiếu file trong AUDIT_ZIP: {missing}'

print('Đầu vào hợp lệ.')


Đầu vào hợp lệ.


In [20]:

# =========================
# 4. ĐỌC BÁO CÁO AUDIT
# =========================
with zipfile.ZipFile(AUDIT_ZIP) as z:
    dup_df = pd.read_csv(z.open('exact_duplicates_all.csv'))
    manifest_df = pd.read_csv(z.open('dataset_manifest.csv'))
    class_dist_df = pd.read_csv(z.open('class_distribution.csv'))
    with z.open('audit_summary.json') as f:
        audit_summary = json.load(f)
    with z.open('class_to_idx.json') as f:
        class_to_idx = json.load(f)

print('Audit status:', audit_summary.get('status'))
print('Exact duplicate pairs:', len(dup_df))
print('Manifest rows:', len(manifest_df))
print('Classes:', len(class_to_idx))

display(dup_df.head())


Audit status: REVIEW_REQUIRED
Exact duplicate pairs: 23
Manifest rows: 13570
Classes: 44


,sha256,split_a,class_a,path_a,split_b,class_b,path_b,cross_split,label_conflict
0,07512c856b4d0dfa0af3a303a78b5533f29629b5cc62434c0422a08525b67f82,train,Agaricus augustus,train/Agaricus augustus/Agaricus augustus_121.jpg,train,Agaricus augustus,train/Agaricus augustus/Agaricus augustus_216.jpg,False,False
1,2ff330e5afe6773de8e1e9d769d5b384f4b9d818052920519ffd67eaeb36513d,train,Agaricus augustus,train/Agaricus augustus/Agaricus augustus_160.jpg,train,Agaricus augustus,train/Agaricus augustus/Agaricus augustus_232.jpg,False,False
2,5f19c1d0a4ce4c8a59439798f7d603870d7c40e92ff87f98a225be857c8da0fe,train,Amanita calyptroderma,train/Amanita calyptroderma/Amanita calyptroderma_175.jpg,train,Amanita calyptroderma,train/Amanita calyptroderma/Amanita calyptroderma_201.jpg,False,False
3,8f81eaa88fd40e50eab73d0c0e0bd9b5258ca1a427465535f11a63cdb219d6bb,train,Amanita flavoconia,train/Amanita flavoconia/Amanita flavoconia_105.jpg,train,Amanita flavoconia,train/Amanita flavoconia/Amanita flavoconia_286.jpg,False,False
4,ab6a680995a55ea3d471b8be1a657483b41a4c025d3b8f862d6e697c81159899,train,Amanita velosa,train/Amanita velosa/Amanita velosa_216.jpg,train,Amanita velosa,train/Amanita velosa/Amanita velosa_256.jpg,False,False


In [21]:

# =========================
# 5. XÁC NHẬN CÁC GIẢ ĐỊNH AN TOÀN
# =========================
assert len(dup_df) == 23, f'Kỳ vọng 23 cặp exact duplicate, nhưng nhận được {len(dup_df)}'
assert (dup_df['cross_split'] == False).all(), 'Có exact duplicate cross-split; cần xử lý đặc biệt.'
assert (dup_df['label_conflict'] == False).all(), 'Có xung đột nhãn trong exact duplicate; cần xử lý thủ công.'
assert (dup_df['split_a'] == dup_df['split_b']).all(), 'Có cặp duplicate nằm khác split.'
assert (dup_df['class_a'] == dup_df['class_b']).all(), 'Có cặp duplicate khác lớp.'

print('Tất cả 23 cặp duplicate đều nội-split và cùng nhãn -> có thể dedup an toàn.')


Tất cả 23 cặp duplicate đều nội-split và cùng nhãn -> có thể dedup an toàn.



## Chiến lược giữ/loại ảnh

Với mỗi nhóm SHA256, notebook sẽ:
1. gom tất cả path thuộc cùng hash,
2. sắp xếp theo thứ tự từ điển,
3. **giữ ảnh đầu tiên** (`keep_path`),
4. **loại các ảnh còn lại** (`drop_path`).

Cách làm này đảm bảo quy tắc nhất quán, tái lập được và không phụ thuộc ngẫu nhiên.


In [22]:

# =========================
# 6. TẠO KẾ HOẠCH DEDUP
# =========================
sha_to_paths = defaultdict(set)
sha_to_split = {}
sha_to_class = {}

for _, row in dup_df.iterrows():
    sha = row['sha256']
    sha_to_paths[sha].add(row['path_a'])
    sha_to_paths[sha].add(row['path_b'])
    sha_to_split[sha] = row['split_a']
    sha_to_class[sha] = row['class_a']

rows = []
for sha, paths in sha_to_paths.items():
    sorted_paths = sorted(paths)
    keep_path = sorted_paths[0]
    drop_paths = sorted_paths[1:]
    for dp in drop_paths:
        rows.append({
            'sha256': sha,
            'split': sha_to_split[sha],
            'class_name': sha_to_class[sha],
            'keep_path': keep_path,
            'drop_path': dp,
            'num_files_in_group': len(sorted_paths),
        })

dedup_plan_df = pd.DataFrame(rows).sort_values(['split', 'class_name', 'keep_path', 'drop_path']).reset_index(drop=True)
removed_files_df = dedup_plan_df[['drop_path', 'split', 'class_name', 'sha256', 'keep_path']].rename(columns={'drop_path': 'relative_path'})
kept_files_df = dedup_plan_df[['keep_path', 'split', 'class_name', 'sha256']].rename(columns={'keep_path': 'relative_path'}).drop_duplicates().reset_index(drop=True)

print('Số nhóm exact duplicate:', len(sha_to_paths))
print('Số file sẽ loại khỏi phiên bản deduped:', len(removed_files_df))
print('Số file đại diện được giữ:', len(kept_files_df))

display(dedup_plan_df.head(10))


Số nhóm exact duplicate: 23
Số file sẽ loại khỏi phiên bản deduped: 23
Số file đại diện được giữ: 23


,sha256,split,class_name,keep_path,drop_path,num_files_in_group
0,700f774e6b90f358fa775cf776bd327b0c78f10696880e36efd424f321cf51d5,test,Armillaria mellea,test/Armillaria mellea/Armillaria mellea_77.jpg,test/Armillaria mellea/Armillaria mellea_94.jpg,2
1,07512c856b4d0dfa0af3a303a78b5533f29629b5cc62434c0422a08525b67f82,train,Agaricus augustus,train/Agaricus augustus/Agaricus augustus_121.jpg,train/Agaricus augustus/Agaricus augustus_216.jpg,2
2,2ff330e5afe6773de8e1e9d769d5b384f4b9d818052920519ffd67eaeb36513d,train,Agaricus augustus,train/Agaricus augustus/Agaricus augustus_160.jpg,train/Agaricus augustus/Agaricus augustus_232.jpg,2
3,5f19c1d0a4ce4c8a59439798f7d603870d7c40e92ff87f98a225be857c8da0fe,train,Amanita calyptroderma,train/Amanita calyptroderma/Amanita calyptroderma_175.jpg,train/Amanita calyptroderma/Amanita calyptroderma_201.jpg,2
4,8f81eaa88fd40e50eab73d0c0e0bd9b5258ca1a427465535f11a63cdb219d6bb,train,Amanita flavoconia,train/Amanita flavoconia/Amanita flavoconia_105.jpg,train/Amanita flavoconia/Amanita flavoconia_286.jpg,2
5,ab6a680995a55ea3d471b8be1a657483b41a4c025d3b8f862d6e697c81159899,train,Amanita velosa,train/Amanita velosa/Amanita velosa_216.jpg,train/Amanita velosa/Amanita velosa_256.jpg,2
6,cb5a30c2c6dcd77bc1ea02b33619d31aa807837f74680eb0f5d43dd005c45404,train,Armillaria mellea,train/Armillaria mellea/Armillaria mellea_107.jpg,train/Armillaria mellea/Armillaria mellea_253.jpg,2
7,dc68d39eecd2496748088b0f484dd35efa1a1d31a2cf2f2991febdd55c39cff7,train,Artomyces pyxidatus,train/Artomyces pyxidatus/Artomyces pyxidatus_20.jpg,train/Artomyces pyxidatus/Artomyces pyxidatus_243.jpg,2
8,b5067ef67ea93a36b77ee7d15dfa373c751a65783584255a2cfc01b594e1ca9b,train,Boletus rex-veris,train/Boletus rex-veris/Boletus rex-veris_158.jpg,train/Boletus rex-veris/Boletus rex-veris_177.jpg,2
9,cda7ef4068fe0d8f0b9683d1e334ea4ad9e9874ca37f409dd0085e364da69c7f,train,Boletus rex-veris,train/Boletus rex-veris/Boletus rex-veris_216.jpg,train/Boletus rex-veris/Boletus rex-veris_286.jpg,2


In [23]:

# =========================
# 7. LƯU BÁO CÁO KẾ HOẠCH
# =========================
dedup_plan_path = REPORT_DIR / 'dedup_plan.csv'
removed_files_path = REPORT_DIR / 'removed_files.csv'
kept_files_path = REPORT_DIR / 'kept_files.csv'

dedup_plan_df.to_csv(dedup_plan_path, index=False)
removed_files_df.to_csv(removed_files_path, index=False)
kept_files_df.to_csv(kept_files_path, index=False)

print('Đã lưu:')
print('-', dedup_plan_path)
print('-', removed_files_path)
print('-', kept_files_path)


Đã lưu:
- /kaggle/working/Mushroom44_Step0_6_DedupExact/reports/dedup_plan.csv
- /kaggle/working/Mushroom44_Step0_6_DedupExact/reports/removed_files.csv
- /kaggle/working/Mushroom44_Step0_6_DedupExact/reports/kept_files.csv


In [24]:

# =========================
# 8. TẠO MANIFEST MỚI SAU DEDUP
# =========================
remove_set = set(removed_files_df['relative_path'].tolist())
manifest_dedup_df = manifest_df[~manifest_df['relative_path'].isin(remove_set)].copy().reset_index(drop=True)

original_count = len(manifest_df)
dedup_count = len(manifest_dedup_df)
removed_count = original_count - dedup_count

print('Original manifest rows:', original_count)
print('Deduped manifest rows :', dedup_count)
print('Removed rows          :', removed_count)
assert removed_count == 23, f'Kỳ vọng loại 23 file, nhưng đang loại {removed_count}'

manifest_dedup_path = REPORT_DIR / 'dataset_manifest_deduped.csv'
manifest_dedup_df.to_csv(manifest_dedup_path, index=False)
print('Đã lưu:', manifest_dedup_path)

display(manifest_dedup_df.head())


Original manifest rows: 13570
Deduped manifest rows : 13547
Removed rows          : 23
Đã lưu: /kaggle/working/Mushroom44_Step0_6_DedupExact/reports/dataset_manifest_deduped.csv


,split,class_name,class_id,relative_path,file_name,file_size_bytes,mtime_ns,image_ok,error_type,image_mode,width,height,aspect_ratio,sha256,phash64
0,train,Agaricus augustus,0,train/Agaricus augustus/Agaricus augustus_1.jpg,Agaricus augustus_1.jpg,49011,1794279903382000000,True,NaN,RGB,640,480,1.333333,14ae2d47227fa89b18b632cf00ba6c417dbb86645b8627c17fe923d6cddc79fb,96ca589d695969d8
1,train,Agaricus augustus,0,train/Agaricus augustus/Agaricus augustus_10.jpg,Agaricus augustus_10.jpg,66730,1794279903044000000,True,NaN,RGB,478,640,0.746875,e126706eb5208040e69c5b4db0407a5a1c0244ad2c792455310effff63da1b6d,81af6e70b51acac5
2,train,Agaricus augustus,0,train/Agaricus augustus/Agaricus augustus_100.jpg,Agaricus augustus_100.jpg,68644,1794279903375000000,True,NaN,RGB,640,480,1.333333,e5259577e0af60ea7eb44ff3f64a79e6c358b3487744e06f98c1175696c2ca94,90953e6bdaf0654a
3,train,Agaricus augustus,0,train/Agaricus augustus/Agaricus augustus_101.jpg,Agaricus augustus_101.jpg,46040,1794279903039000000,True,NaN,RGB,480,640,0.750000,294965c9f281828dcdd24085c89b0a5c34337dee25a86977bc3c6922e80f0f13,9916673996ce7930
4,train,Agaricus augustus,0,train/Agaricus augustus/Agaricus augustus_102.jpg,Agaricus augustus_102.jpg,37439,1794279903032000000,True,NaN,RGB,640,480,1.333333,bfb32ea2176d01f41db0aa4bdc2ba3c5ba8c40b8ec03ec3b3bf03cc5b575d48f,e4e721fa332f18e0


In [25]:

# =========================
# 9. THỐNG KÊ PHÂN BỐ LỚP MỚI
# =========================
# Cố gắng nhận biết tên cột split / class trong manifest
possible_split_cols = ['split', 'subset']
possible_class_cols = ['class_name', 'class', 'label_name']

split_col = next((c for c in possible_split_cols if c in manifest_dedup_df.columns), None)
class_col = next((c for c in possible_class_cols if c in manifest_dedup_df.columns), None)

assert split_col is not None, f'Không tìm thấy cột split trong manifest: {manifest_dedup_df.columns.tolist()}'
assert class_col is not None, f'Không tìm thấy cột class trong manifest: {manifest_dedup_df.columns.tolist()}'

class_distribution_dedup_df = (
    manifest_dedup_df.groupby([split_col, class_col])
    .size()
    .reset_index(name='count')
    .sort_values([split_col, class_col])
    .reset_index(drop=True)
)

class_distribution_dedup_path = REPORT_DIR / 'class_distribution_deduped.csv'
class_distribution_dedup_df.to_csv(class_distribution_dedup_path, index=False)

print('Đã lưu:', class_distribution_dedup_path)
display(class_distribution_dedup_df.head(12))


Đã lưu: /kaggle/working/Mushroom44_Step0_6_DedupExact/reports/class_distribution_deduped.csv


,split,class_name,count
0,test,Agaricus augustus,51
1,test,Amanita amerirubescens,49
2,test,Amanita brunnescens,47
3,test,Amanita calyptroderma,49
4,test,Amanita flavoconia,48
5,test,Amanita muscaria,49
6,test,Amanita velosa,43
7,test,Armillaria mellea,48
8,test,Armillaria tabescens,51
9,test,Artomyces pyxidatus,51


In [26]:

# =========================
# 10. TỔNG HỢP THỐNG KÊ TRƯỚC/SAU DEDUP
# =========================
orig_split_counts = manifest_df[split_col].value_counts().to_dict()
dedup_split_counts = manifest_dedup_df[split_col].value_counts().to_dict()
removed_by_split = removed_files_df['split'].value_counts().to_dict()

summary = {
    'dataset_root_original': str(DATASET_ROOT),
    'audit_zip': str(AUDIT_ZIP),
    'original_total_images': int(len(manifest_df)),
    'deduped_total_images': int(len(manifest_dedup_df)),
    'removed_exact_duplicates': int(len(removed_files_df)),
    'original_split_counts': {k: int(v) for k, v in orig_split_counts.items()},
    'deduped_split_counts': {k: int(v) for k, v in dedup_split_counts.items()},
    'removed_by_split': {k: int(v) for k, v in removed_by_split.items()},
    'num_classes': int(len(class_to_idx)),
    'near_duplicate_cross_split_status': 'reviewed_by_user_keep_both',
    'decision_rule': 'keep lexicographically first path within each exact-duplicate SHA256 group; remove all others',
    'status': 'PASS_FOR_TRAINING_AFTER_DEDUP'
}

summary_path = REPORT_DIR / 'dedup_summary.json'
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(json.dumps(summary, indent=2, ensure_ascii=False))
print('Đã lưu:', summary_path)


{
  "dataset_root_original": "/kaggle/input/datasets/thoduynhunh/mushroom-44-clean",
  "audit_zip": "/kaggle/working/Mushroom44_Step0_DatasetAudit.zip",
  "original_total_images": 13570,
  "deduped_total_images": 13547,
  "removed_exact_duplicates": 23,
  "original_split_counts": {
    "train": 9497,
    "test": 2065,
    "valid": 2008
  },
  "deduped_split_counts": {
    "train": 9476,
    "test": 2064,
    "valid": 2007
  },
  "removed_by_split": {
    "train": 21,
    "test": 1,
    "valid": 1
  },
  "num_classes": 44,
  "near_duplicate_cross_split_status": "reviewed_by_user_keep_both",
  "decision_rule": "keep lexicographically first path within each exact-duplicate SHA256 group; remove all others",
  "status": "PASS_FOR_TRAINING_AFTER_DEDUP"
}
Đã lưu: /kaggle/working/Mushroom44_Step0_6_DedupExact/reports/dedup_summary.json


In [27]:

# =========================
# 11. TẠO DATASET DEDUPED MỚI (KHÔNG SỬA DỮ LIỆU GỐC)
# =========================
def safe_link_or_copy(src: Path, dst: Path, mode='symlink'):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        return
    if mode == 'symlink':
        try:
            os.symlink(src, dst)
            return
        except Exception:
            shutil.copy2(src, dst)
            return
    elif mode == 'copy':
        shutil.copy2(src, dst)
    else:
        raise ValueError(f'BUILD_MODE không hợp lệ: {mode}')

if BUILD_DEDUPED_DATASET:
    if OUTPUT_DATASET_ROOT.exists():
        shutil.rmtree(OUTPUT_DATASET_ROOT)
    OUTPUT_DATASET_ROOT.mkdir(parents=True, exist_ok=True)

    copied = 0
    missing = []
    for rel_path in manifest_dedup_df['relative_path'].tolist():
        src = DATASET_ROOT / rel_path
        dst = OUTPUT_DATASET_ROOT / rel_path
        if not src.exists():
            missing.append(rel_path)
            continue
        safe_link_or_copy(src, dst, mode=BUILD_MODE)
        copied += 1

    print(f'Đã tạo dataset deduped tại: {OUTPUT_DATASET_ROOT}')
    print(f'Số file đã materialize: {copied}')
    print(f'Số file thiếu: {len(missing)}')

    if missing:
        missing_path = REPORT_DIR / 'missing_files_when_building_dataset.csv'
        pd.DataFrame({'relative_path': missing}).to_csv(missing_path, index=False)
        print('Danh sách file thiếu đã lưu:', missing_path)

    assert copied == len(manifest_dedup_df) - len(missing), 'Số file tạo ra không khớp manifest deduped.'
else:
    print('Bỏ qua bước tạo dataset deduped vật lý (BUILD_DEDUPED_DATASET=False).')


Đã tạo dataset deduped tại: /kaggle/working/Mushroom44_Step0_6_DedupExact/mushroom-44-clean-deduped
Số file đã materialize: 13547
Số file thiếu: 0


In [28]:

# =========================
# 12. KIỂM TRA NHANH CÂY THƯ MỤC MỚI
# =========================
if BUILD_DEDUPED_DATASET:
    split_counts_actual = {}
    for split in ['train', 'valid', 'test']:
        split_dir = OUTPUT_DATASET_ROOT / split
        count = sum(1 for p in split_dir.rglob('*') if p.is_file() or p.is_symlink()) if split_dir.exists() else 0
        split_counts_actual[split] = count

    print('Số file thực tế trong dataset deduped:', split_counts_actual)

    expected = summary['deduped_split_counts']
    print('Số file kỳ vọng theo manifest deduped:', expected)


Số file thực tế trong dataset deduped: {'train': 9476, 'valid': 2007, 'test': 2064}
Số file kỳ vọng theo manifest deduped: {'train': 9476, 'test': 2064, 'valid': 2007}


In [29]:

# =========================
# 13. ZIP TOÀN BỘ KẾT QUẢ
# =========================
zip_output = WORK_ROOT / 'Mushroom44_Step0_6_DedupExact.zip'
if zip_output.exists():
    zip_output.unlink()

with zipfile.ZipFile(zip_output, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    # reports
    for p in REPORT_DIR.rglob('*'):
        if p.is_file():
            z.write(p, arcname=str(p.relative_to(WORK_ROOT)))
    # class mapping
    class_to_idx_path = REPORT_DIR / 'class_to_idx.json'
    with open(class_to_idx_path, 'w', encoding='utf-8') as f:
        json.dump(class_to_idx, f, indent=2, ensure_ascii=False)
    z.write(class_to_idx_path, arcname=str(class_to_idx_path.relative_to(WORK_ROOT)))

print('Đã tạo ZIP:', zip_output)


Đã tạo ZIP: /kaggle/working/Mushroom44_Step0_6_DedupExact/Mushroom44_Step0_6_DedupExact.zip



## Kết quả mong đợi sau khi chạy xong

Nếu mọi thứ đúng như audit hiện tại, bạn sẽ có:
- loại bỏ đúng **23 file trùng tuyệt đối**,
- giữ nguyên **44 lớp**,
- số lượng ảnh mới dự kiến:
  - **train:** 9476
  - **valid:** 2007
  - **test:** 2064
  - **tổng:** 13547

## Sau notebook này, bạn làm gì tiếp?
1. Gửi cho tôi file ZIP `Mushroom44_Step0_6_DedupExact.zip` (hoặc ít nhất là `dedup_summary.json`, `removed_files.csv`, `dataset_manifest_deduped.csv`).
2. Sau khi tôi xác nhận manifest mới, tôi sẽ tạo notebook **Step 1 — Train B0, B1, B2, B3, B4** với seed 42 và `RUN_FINAL_TEST=False`.
